## Initial Imports

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

# print(Path.cwd())
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT)) # Tells python to check project root first when looking for imports (ie below)

In [ ]:
from src.data_loader import load_movie_details, load_reviews

## Load data

In [ ]:
reviews = load_reviews()
movies = load_movie_details()

## Basic Info + Check for Invalid Data

### Identify Missing Value(s)

In [ ]:
reviews.info()
movies.info()

In [ ]:
def missing_value_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize pandas-recognized missing values in each column."""
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing_count": df.isna().sum(),
            "missing_percent": (
                df.isna().mean().mul(100).round(2)
            ),
        }
    ).sort_values("missing_count", ascending=False)

reviews_missing = missing_value_summary(reviews)
movies_missing = missing_value_summary(movies)

display(reviews_missing)
display(movies_missing)

In [ ]:
for column in ["review_text", "review_summary"]:
    blank_mask = reviews[column].map(
        lambda value: (
            isinstance(value, str)
            and not value.strip()
        )
    )

    print(
        f"{column}: "
        f"{blank_mask.sum():,} blank strings"
    )

No pandas-recognized missing values were found in either dataset. However, two `review_summary` values are blank strings.

### Verify value types

In [ ]:
for column in ["plot_summary", "plot_synopsis", "genre"]:
    print(f"\nValue types in {column}:")

    display(
        movies[column]
        .map(lambda value: type(value).__name__)
        .value_counts(dropna=False)
    )

    display(movies[column].head(3))

#### Investigation Into Repeated Review Text

In [ ]:
duplicate_review_mask = reviews.duplicated(
    subset=["movie_id", "review_text"],
    keep=False,
)

print(
    "Rows sharing the same movie_id and review_text:",
    f"{duplicate_review_mask.sum():,}",
)

display(
    reviews.loc[
        duplicate_review_mask,
        ["movie_id", "user_id", "review_text", "is_spoiler"],
    ].head(10)
)

##### Identify Exact Duplicates

First, check whether any rows are exact duplicates across all columns.

An exact duplicate would mean that fields such as the movie, user, review text,
date, rating, and spoiler label are all identical.

In [ ]:
exact_duplicate_mask = reviews.duplicated(keep=False)

exact_duplicate_count = exact_duplicate_mask.sum()

print(f"Rows involved in exact duplicate records: {exact_duplicate_count:,}")

**Finding:** No exact duplicate records were found.

Therefore, subsequent duplicate analysis focuses on repeated review text rather than fully duplicated rows.

##### Repeated review text within the same movie

Although there are no exact duplicate rows, the same review text may appear multiple times for the same movie.

The following analysis groups records by `movie_id` and `review_text` and investigates how often these repeated-text groups occur.

In [ ]:
repeated_review_groups = (
    reviews
    .groupby(["movie_id", "review_text"])
    .agg(
        row_count=("user_id", "size"),
        unique_users=("user_id", "nunique"),
        unique_dates=("review_date", "nunique"),
        unique_ratings=("rating", "nunique"),
        unique_labels=("is_spoiler", "nunique"),
    )
    .query("row_count > 1")
    .sort_values("row_count", ascending=False)
)

In [ ]:
num_repeated_groups = len(repeated_review_groups)
num_rows_in_repeated_groups = repeated_review_groups["row_count"].sum()

print(f"Repeated-text groups: {num_repeated_groups:,}")
print(f"Rows belonging to repeated-text groups: {num_rows_in_repeated_groups:,}")
print(
    "Percentage of all review rows in repeated-text groups: "
    f"{num_rows_in_repeated_groups / len(reviews) * 100:.3f}%"
)

In [ ]:
repeat_count_distribution = (
    repeated_review_groups["row_count"]
    .value_counts()
    .sort_index()
    .rename_axis("times_same_text_appears")
    .to_frame("number_of_groups")
)

display(repeat_count_distribution)

##### Identify Users/Dates/Ratings Associated With Repeated Review Text

If repeated text belongs to the same user/date/rating, it may represent repeated submissions.
If identical text is associated with multiple users/dates/ratings, it may instead represent copied reviews or some other characteristic of the source dataset.

In [ ]:
same_user_groups = (
    repeated_review_groups["unique_users"] == 1
)

multiple_user_groups = (
    repeated_review_groups["unique_users"] > 1
)

print(
    "Repeated-text groups belonging to only one user:",
    f"{same_user_groups.sum():,}"
)

print(
    "Repeated-text groups involving multiple users:",
    f"{multiple_user_groups.sum():,}"
)

In [ ]:
same_date_groups = (
    repeated_review_groups["unique_dates"] == 1
)

multiple_date_groups = (
    repeated_review_groups["unique_dates"] > 1
)

print(
    "Repeated-text groups occurring on one date:",
    f"{same_date_groups.sum():,}"
)

print(
    "Repeated-text groups occurring across multiple dates:",
    f"{multiple_date_groups.sum():,}"
)

In [ ]:
conflicting_rating_groups = (
    repeated_review_groups["unique_ratings"] > 1
)

print(
    "Repeated-text groups containing different ratings:",
    f"{conflicting_rating_groups.sum():,}"
)

print(
    "Percentage of repeated-text groups with different ratings:",
    f"{conflicting_rating_groups.mean() * 100:.2f}%"
)

##### Conflicting spoiler labels

A particularly important case occurs when identical review text for the same
movie is labelled both as a spoiler and as a non-spoiler.

For text-based classification, this means the same model input can be associated
with contradictory target labels.

In [ ]:
conflicting_label_groups = (
    repeated_review_groups[
        repeated_review_groups["unique_labels"] > 1
    ]
)

num_conflicting_groups = len(conflicting_label_groups)

num_rows_in_conflicting_groups = (
    conflicting_label_groups["row_count"].sum()
)

print(
    "Repeated-text groups with conflicting spoiler labels:",
    f"{num_conflicting_groups:,}"
)

print(
    "Rows belonging to conflicting-label groups:",
    f"{num_rows_in_conflicting_groups:,}"
)

print(
    "Percentage of repeated-text groups with conflicting labels:",
    f"{num_conflicting_groups / len(repeated_review_groups) * 100:.2f}%"
)

In [ ]:
display(
    conflicting_label_groups
    .reset_index()
    .sort_values("row_count", ascending=False)
    .head(10)[
        [
            "movie_id",
            "review_text",
            "row_count",
            "unique_users",
            "unique_dates",
            "unique_ratings",
            "unique_labels",
        ]
    ]
)

In [ ]:
def get_repeated_review_records(movie_id, review_text):
    """Return all rows for a repeated movie-review text group."""
    
    records = reviews[
        (reviews["movie_id"] == movie_id)
        & (reviews["review_text"] == review_text)
    ][
        [
            "movie_id",
            "user_id",
            "review_date",
            "rating",
            "is_spoiler",
            "review_summary",
            "review_text",
        ]
    ].copy()

    records["review_date"] = pd.to_datetime(
        records["review_date"],
        format="%d %B %Y",
        errors="coerce",
    )

    return records.sort_values("review_date")

In [ ]:
movie_id, review_text = conflicting_label_groups.index[0]

display(
    get_repeated_review_records(
        movie_id,
        review_text,
    )
)

In [ ]:
for movie_id, review_text in conflicting_label_groups.index[:3]:
    print(f"\nMovie: {movie_id}")

    display(
        get_repeated_review_records(
            movie_id,
            review_text,
        )
    )

###### Summarize Characteristics of Repeated Reviews

In [ ]:
repeated_review_characteristics = pd.DataFrame(
    {
        "category": [
            "Repeated-text groups",
            "Multiple users",
            "Multiple dates",
            "Different ratings",
            "Conflicting spoiler labels",
        ],
        "group_count": [
            len(repeated_review_groups),
            (repeated_review_groups["unique_users"] > 1).sum(),
            (repeated_review_groups["unique_dates"] > 1).sum(),
            (repeated_review_groups["unique_ratings"] > 1).sum(),
            (repeated_review_groups["unique_labels"] > 1).sum(),
        ],
    }
)

repeated_review_characteristics["percent_of_repeated_groups"] = (
    repeated_review_characteristics["group_count"]
    / len(repeated_review_groups)
    * 100
).round(2)

display(repeated_review_characteristics)

In [ ]:
text_across_movies = (
    reviews
    .groupby("review_text")
    .agg(
        row_count=("movie_id", "size"),
        unique_movies=("movie_id", "nunique"),
        unique_users=("user_id", "nunique"),
        unique_labels=("is_spoiler", "nunique"),
    )
    .query("unique_movies > 1")
    .sort_values(
        ["unique_movies", "row_count"],
        ascending=False,
    )
)

print(
    "Review texts appearing across multiple movies:",
    f"{len(text_across_movies):,}"
)

display(text_across_movies.head(10))

##### Summary of Investigation

No exact duplicate rows were found.

However, 357 groups contain identical review text for the same movie,
affecting 786 of 573,913 review records (0.137%).

These should not be treated as conventional duplicate records:

- all 357 groups involve multiple users;
- 350 groups occur across multiple review dates;
- 180 groups contain differing ratings;
- 69 groups contain conflicting spoiler labels.

The repeated records therefore appear to represent separate records containing reused review text rather than accidental duplicate rows.

This has implications for model evaluation. If identical review text appears in both training and test sets, a text classifier may receive effectively the same input during training and evaluation, artificially inflating measured performance. As such, it is recommended that identical review text should not appear across train/validation/test splits.

Because these records represent only a very small fraction of the dataset, they do not substantially reduce the available sample size if special handling is required during model preparation.

## Broader Exploratory Data Analysis

The previous sections focused primarily on data-quality issues. The following
sections examine the overall characteristics of the review and movie datasets
that are relevant to the project's research questions.

### Spoiler Label Distribution

We would like to examine the balance between spoiler and non-spoiler reviews.

This is important because substantial class imbalance can make metrics such as accuracy misleading and affects how model performance should be evaluated.

In [ ]:
label_counts = reviews["is_spoiler"].value_counts().sort_index()
label_percentages = (
    reviews["is_spoiler"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

label_distribution = pd.DataFrame({
    "count": label_counts,
    "percentage": label_percentages.round(2),
})

label_distribution.index = [
    "Non-spoiler",
    "Spoiler",
]

display(label_distribution)

**Finding:** 73.7% of reviews are labelled as non-spoilers and 26.3% are spoilers.

This indicates that the dataset is imbalanced, with non-spoilers forming the majority class. The class distribution should be taken into consideration when selecting evaluation metrics for the classification models.

### Review Length

We examine review length to understand the amount of text available to the text-based models and whether there is a long tail of unusually large reviews.

Word count here is an approximate count based on whitespace-separated tokens.

In [ ]:
review_word_count = (
    reviews["review_text"]
    .str.count(r"\S+")
)

display(
    review_word_count.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

In [ ]:
upper_limit = review_word_count.quantile(0.99)

review_word_count[
    review_word_count <= upper_limit
].plot(
    kind="hist",
    bins=50,
    title="Distribution of Review Lengths (up to 99th percentile)",
)

plt.xlabel("Approximate Word Count")
plt.ylabel("Number of Reviews")
plt.show()

**Finding:** The median review contains approximately 189 words, while the 95th and 99th percentiles are approximately 695 and 960 words respectively.

The distribution is right-skewed. The long-review tail may be relevant later when considering computational requirements for text and embedding-based models.

### Reviews Per Movie

We examine the number of reviews associated with each movie to determine whether reviews are evenly distributed across movies or concentrated among a smaller set of popular movies.

This is particularly relevant to RQ3, where evaluation will use a movie-disjoint split.

In [ ]:
reviews_per_movie = reviews["movie_id"].value_counts()

display(
    reviews_per_movie.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

In [ ]:
display(
    reviews_per_movie
    .head(10)
    .rename("review_count")
    .to_frame()
)

In [ ]:
reviews_per_movie.head(10).sort_values().plot(
    kind="barh",
    title="Movies With the Most Reviews",
)

plt.xlabel("Number of Reviews")
plt.ylabel("Movie ID")
plt.show()

**Finding:** The dataset contains reviews from 1572 unique movies. The median movie has 326 reviews, while the most-reviewed movies contain substantially more.

This uneven distribution motivates the RQ3 movie-disjoint evaluation: a conventional row-level split can contain reviews of the same movie in both training and evaluation data.

### Plot Context Availability

RQ2 requires movie-specific plot information. Although no null values were identified in the movie metadata, plot fields may contain blank strings.

The following analysis measures the availability of usable plot summaries and plot synopses.

In [ ]:
plot_availability = pd.DataFrame({
    "blank_count": [
        movies["plot_summary"].str.strip().eq("").sum(),
        movies["plot_synopsis"].str.strip().eq("").sum(),
    ],
    "available_count": [
        movies["plot_summary"].str.strip().ne("").sum(),
        movies["plot_synopsis"].str.strip().ne("").sum(),
    ],
}, index=["plot_summary", "plot_synopsis"])

plot_availability["available_percentage"] = (
    plot_availability["available_count"]
    / len(movies)
    * 100
).round(2)

display(plot_availability)

**Finding:** 100% of movies contain a non-empty plot summary and 85.18% contain a non-empty plot synopsis.

This determines how much of the dataset can directly support the RQ2 plot-context comparison.

### Plot Context Length

The lengths of the available plot summaries and synopses are examined because RQ2 will compare review text against movie plot information.

Very long synopses may later require special handling for embedding models with limited input lengths.

In [ ]:
plot_summary_word_count = (
    movies.loc[
        movies["plot_summary"].str.strip().ne(""),
        "plot_summary",
    ]
    .str.count(r"\S+")
)

plot_synopsis_word_count = (
    movies.loc[
        movies["plot_synopsis"].str.strip().ne(""),
        "plot_synopsis",
    ]
    .str.count(r"\S+")
)

In [ ]:
plot_length_summary = pd.DataFrame({
    "plot_summary": plot_summary_word_count.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ),
    "plot_synopsis": plot_synopsis_word_count.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ),
})

display(plot_length_summary)

**Finding:** Plot synopses are much longer than plot summaries.

The observed lengths should be considered when implementing the RQ2 semantic similarity approach, particularly if the selected embedding model imposes an input-length limit.

### Genre Distribution

Movies may belong to multiple genres. The genre lists are therefore expanded so that each genre can be counted individually.

In [ ]:
genre_counts = (
    movies["genre"]
    .explode()
    .value_counts()
)

display(genre_counts)

In [ ]:
genre_counts.head(15).sort_values().plot(
    kind="barh",
    title="Most Common Movie Genres",
)

plt.xlabel("Number of Movies")
plt.ylabel("Genre")
plt.show()

**Finding:** The dataset is concentrated most heavily in Drama movies.

### Explicit Spoiler-Word Mentions

As preliminary context for RQ4, examine how frequently review bodies contain the explicit words `spoiler` or `spoilers`.

This does not represent the final masking rule; it only measures how common these obvious terms are in the raw review text.

In [ ]:
spoiler_word_mask = (
    reviews["review_text"]
    .str.contains(
        r"\bspoilers?\b",
        case=False,
        regex=True,
        na=False,
    )
)

print(
    "Reviews containing 'spoiler' or 'spoilers':",
    f"{spoiler_word_mask.sum():,}",
)

print(
    "Percentage of reviews:",
    f"{spoiler_word_mask.mean() * 100:.2f}%",
)

**Finding:** 4.48% of reviews explicitly contain the word `spoiler` or `spoilers`.

This provides motivation for RQ4, which will later test whether classifier performance depends heavily on such explicit warning language.

### Identify Movies Without Metadata Match

In [ ]:
known_movie_ids = movies["movie_id"].dropna()

missing_metadata_mask = (
    reviews["movie_id"].notna()
    & ~reviews["movie_id"].isin(known_movie_ids)
)

print(
    "Reviews whose movie_id has no metadata match:",
    f"{missing_metadata_mask.sum():,}",
)

display(
    reviews.loc[
        missing_metadata_mask,
        ["movie_id", "review_text"],
    ].head(10)
)

In [ ]:
invalid_movie_ids = movies[
    ~movies["movie_id"]
    .astype("string")
    .str.fullmatch(r"tt\d+")
]

invalid_movie_ids[["movie_id"]]

While it initially seems like there are two unaccounted for movie IDs (tt0104014, tt0114142), it turns out the actual issue is that these movie_id values were filled in inconsistently (with a trailing '/').

## EDA Summary

### Data Quality

- The dataset contains 573,913 reviews and metadata for 1,572 movies.
- No exact duplicate rows were identified.
- 357 same-movie repeated-review-text groups were identified, involving
  786 rows (0.137% of all reviews).
- All repeated-text groups involve multiple users, indicating that these
  are not conventional duplicate rows.
- 69 repeated-text groups contain conflicting spoiler labels, affecting
  181 rows.
- 76 exact review texts also occur across multiple movies.
- No pandas-recognized null values were found. However, 2 `review_summary`
  values are blank strings and 233 `plot_synopsis` values are blank.
- Two movie metadata IDs (`tt0104014/` and `tt0114142/`) contain an
  inconsistent trailing `/`. This causes 7 review records to initially fail
  to match their corresponding movie metadata.

### Dataset Characteristics

- The target labels are imbalanced: 73.7% of reviews are labelled as non-spoilers and 26.3% as spoilers.
- Review length is right-skewed. The median review contains approximately 189 words, while the 95th and 99th percentiles are approximately 695 and 960 words respectively. The longest review contains 2,675 words.
- Reviews are distributed across 1,572 unique movies. The median movie has 326 reviews, while the most-reviewed movie has 4,845 reviews, indicating that some movies are represented much more heavily than others.
- All 1,572 movies contain a non-empty plot summary, while 1,339 movies (85.18%) contain a non-empty plot synopsis.
- Plot synopses are substantially longer than plot summaries. Plot summaries have a median length of 96 words, while synopsis lengths can extend to several thousand words, with a maximum observed length of 11,396 words.
- Drama is the most common genre, appearing in 799 movies, followed by Comedy (525), Action (438), and Adventure (433). Movies may belong to multiple genres.
- 25,699 reviews (4.48%) explicitly contain the word `spoiler` or `spoilers`, providing motivation for the later RQ4 masking experiment.

### Modelling Implications

The repeated-review investigation has implications for how the data should be split for model training and evaluation.

Identical `review_text` values should not appear in both training and evaluation sets. Otherwise, a text-based classifier may see effectively the same input during training and testing, artificially inflating measured performance. For the standard text-based evaluation, records sharing the same exact `review_text` should therefore be assigned to the same split.

For RQ3, reviews belonging to the same `movie_id` must additionally remain within the same split so that evaluation genuinely measures performance on unseen movies. Because some identical review texts occur across different movies, review-text overlap should also be checked after constructing the movie-disjoint split.

The 69 same-movie repeated-text groups with conflicting `is_spoiler` labels represent ambiguous training examples: the same movie and review body are associated with different target labels. These records should be flagged and handled consistently during model preparation rather than assigning one label arbitrarily.

The malformed trailing `/` characters in movie IDs should be normalized before reviews are joined with movie metadata.

For RQ2, plot summaries are available for every movie, while plot synopses are available for approximately 85% of movies. Since some synopses are extremely long, input-length limitations will need to be considered when implementing semantic embedding approaches.

For RQ4, the fact that 4.48% of reviews explicitly contain `spoiler` or `spoilers` provides a basis for testing whether model performance depends on obvious spoiler-warning language.